In [1]:
!pip install groq -q
print("Groq installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.0 MB/s eta 0:00:00
Groq installed!


In [ ]:
from groq import Groq

# Paste your Groq API key here
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = Groq(api_key=GROQ_API_KEY)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",      # free, fast Llama 3 model
    messages=[
        {
            "role": "user",
            "content": "What is Apple Scab disease in plants? Answer in 3 sentences."
        }
    ],
    max_tokens=200
)

# Extract the text reply
reply = response.choices[0].message.content
print(reply)

Apple Scab is a fungal disease that affects apple and other fruit trees, causing yellow or brown spots to form on the leaves, and potentially leading to defoliation. The disease is caused by the fungus Venturia inaequalis, which overwinters on infected leaves and spreads to new growth in the spring through rain or irrigation. If left untreated, Apple Scab can significantly reduce fruit production and overall tree health, making it a significant concern for orchard owners and gardeners.


In [4]:
print("""
REGULAR LLM CALL:
  User → "What is Apple Scab?"
  LLM  → answers from training knowledge
  Done.

LLM AGENT:
  User  → "Diagnose this disease and give treatment"
  Agent → thinks: "I need to call disease_info() tool first"
  Agent → calls disease_info("Apple Scab")
  Tool  → returns structured data
  Agent → uses that data to form a complete response
  Done.

The difference: an agent DECIDES when to call tools.
It reasons about what information it needs before answering.
""")


REGULAR LLM CALL:
  User → "What is Apple Scab?"
  LLM  → answers from training knowledge
  Done.

LLM AGENT:
  User  → "Diagnose this disease and give treatment"
  Agent → thinks: "I need to call disease_info() tool first"
  Agent → calls disease_info("Apple Scab")
  Tool  → returns structured data
  Agent → uses that data to form a complete response
  Done.

The difference: an agent DECIDES when to call tools.
It reasons about what information it needs before answering.



In [6]:
import json
import os

disease_data = {
    "Apple Scab": {
        "cause": "Fungal infection caused by Venturia inaequalis",
        "symptoms": "Dark olive-green spots on leaves, eventually turning brown and scabby",
        "severity": "Medium"
    },
    "Late blight": {
        "cause": "Water mold Phytophthora infestans",
        "symptoms": "Dark water-soaked lesions on leaves and stems, white mold on underside",
        "severity": "High"
    },
    "Early blight": {
        "cause": "Fungal infection by Alternaria solani",
        "symptoms": "Dark brown spots with concentric rings, yellow halo around spots",
        "severity": "Medium"
    },
    "Powdery mildew": {
        "cause": "Various fungal species depending on host plant",
        "symptoms": "White powdery coating on leaf surface, distorted young leaves",
        "severity": "Low"
    },
    "Northern Leaf Blight": {
        "cause": "Fungal infection by Exserohilum turcicum",
        "symptoms": "Long greyish-green lesions on corn leaves, cigar-shaped spots",
        "severity": "Medium"
    },
    "Tomato Yellow Leaf Curl Virus": {
        "cause": "Virus transmitted by whitefly Bemisia tabaci",
        "symptoms": "Yellowing and upward curling of leaves, stunted plant growth",
        "severity": "High"
    },
    "healthy": {
        "cause": "No disease detected",
        "symptoms": "Plant appears healthy with no visible symptoms",
        "severity": "None"
    }
}

# Save to file
os.makedirs("agent", exist_ok=True)
with open("agent/disease_data.json", "w") as f:
    json.dump(disease_data, f, indent=2)

print("disease_data.json saved!")

disease_data.json saved!


In [7]:
import os

def disease_info(disease_name):
    """
    Tool 1: Given a disease name, returns cause + symptoms + severity.
    This is what the agent will call after the model makes a prediction.
    """
    with open("agent/disease_data.json", "r") as f:
        data = json.load(f)

    # Try exact match first
    if disease_name in data:
        info = data[disease_name]
        return {
            "disease": disease_name,
            "cause":    info["cause"],
            "symptoms": info["symptoms"],
            "severity": info["severity"],
            "found":    True
        }

    # Try partial match — handles "Apple___Apple_Scab" style names
    for key in data:
        if key.lower() in disease_name.lower() or disease_name.lower() in key.lower():
            info = data[key]
            return {
                "disease": key,
                "cause":    info["cause"],
                "symptoms": info["symptoms"],
                "severity": info["severity"],
                "found":    True
            }

    # Not found
    return {
        "disease": disease_name,
        "cause":    "Information not available",
        "symptoms": "Information not available",
        "severity": "Unknown",
        "found":    False
    }

# Test it
result = disease_info("Apple Scab")
print(json.dumps(result, indent=2))

{
  "disease": "Apple Scab",
  "cause": "Fungal infection caused by Venturia inaequalis",
  "symptoms": "Dark olive-green spots on leaves, eventually turning brown and scabby",
  "severity": "Medium",
  "found": true
}


In [9]:
def run_agent(disease_name):
    """
    Agent workflow:
    1. Call disease_info tool to get structured data
    2. Pass that data to Groq LLM
    3. LLM formats a complete expert response
    """

    # Step 1 — call the tool
    tool_result = disease_info(disease_name)

    # Step 2 — build prompt with tool result injected
    system_prompt = """You are an expert agricultural advisor helping farmers
diagnose and treat crop diseases. Be concise, practical, and helpful.
Always structure your response clearly."""

    user_message = f"""
A farmer's crop has been diagnosed with: {tool_result['disease']}

Here is the disease information:
- Cause: {tool_result['cause']}
- Symptoms: {tool_result['symptoms']}
- Severity: {tool_result['severity']}

Give the farmer:
1. A brief explanation of what this disease is
2. How serious it is
3. What they should do immediately
Keep it under 150 words.
"""

    # Step 3 — call Groq
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ],
        max_tokens=300
    )

    return response.choices[0].message.content

# Test it
print(run_agent("Apple Scab"))

**Disease Overview**
Apple Scab is a fungal infection caused by Venturia inaequalis, resulting in dark olive-green spots on leaves that turn brown and scabby.

**Severity**
The disease is considered medium severity, meaning it can cause significant damage but is not typically catastrophic.

**Immediate Action**
Remove and dispose of infected leaves to prevent further spread. Apply a fungicide specifically labeled for Apple Scab control, following the product's instructions. Ensure good air circulation and maintain a regular sanitation schedule to reduce the risk of further infection. Monitor the crop closely for any further signs of disease.


In [10]:
test_diseases = ["Late blight", "Powdery mildew", "healthy"]

for disease in test_diseases:
    print(f"\n{'='*60}")
    print(f"Disease: {disease}")
    print('='*60)
    print(run_agent(disease))


Disease: Late blight
**Disease Explanation**: Late blight is a fungal-like disease caused by Phytophthora infestans, affecting leaves and stems.

**Severity**: It's a highly severe disease that can quickly spread, causing significant damage and crop loss.

**Immediate Action**: Remove and destroy infected plants to prevent further spread. Apply a fungicide specifically labeled for late blight control, and ensure good air circulation and water management practices to reduce moisture. Monitor the crop closely for additional symptoms and consider seeking guidance on integrated pest management strategies to minimize disease impact. Act quickly to minimize damage and potential crop loss.

Disease: Powdery mildew
**Disease Overview**
Powdery mildew is a fungal disease causing a white, powdery coating on leaf surfaces and distorting young leaves.

**Severity**
The disease is currently at a low severity level, but it can spread if left unmanaged.

**Immediate Action**
Immediately remove and d